# Broadened SAC set with UMA: N/C divacancy and single-vacancy sites

**280 new catalysts** = 28 metals × 10 new coordination environments:

* **Family A** (4-coordinate, divacancy): N3C1, N2C2-PEN, N2C2-HEX, N2C2-OPP, N1C3, C4
* **Family C** (3-coordinate, single vacancy): SV-N3, SV-N2C1, SV-N1C2, SV-C3

**Read this first: what these numbers are worth.** The validation run on Xu's 23 catalysts showed that
*zero-shot* UMA is **not** DFT-quality for NRR intermediates. For ΔE of *N2, *NNH and *NH2 it was no better than
guessing the average (MAE 0.57–0.85 eV, r ≤ 0.6).

It did carry some trend for ΔG*H (r ≈ 0.72) and the stability Ef (r ≈ 0.78). So this notebook is used to:

1. **relax the structures:** UMA geometries are good, cheap starting points for DFT. They cut DFT cost a lot.
2. **give rough Ef / ΔG*H trends**, as a first filter, clearly labelled *UMA zero-shot*;
3. later, **run a fine-tuned UMA** (after DFT data for the catalysts in `dft_request_candidates.csv` exists). Only
   then are the NRR energies trustworthy.

Same mechanics as the first notebook: results go to Google Drive after every structure, and a re-run resumes.

## 1. GPU, install, login (same as before)

In [ ]:
import sys, os, subprocess, warnings
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules
MODEL = 'uma'          # 'uma' | 'mace' (no-login stand-in for testing)
if IN_COLAB:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv'], capture_output=True, text=True).stdout)
    pkgs = ['fairchem-core'] if MODEL == 'uma' else ['mace-torch']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
if MODEL == 'uma':
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            pass
    from huggingface_hub import login
    login(token=token) if token else login()

## 2. Files
Upload **`SAC_broadened_package.zip`** into `My Drive/NRR_UMA` (the same folder as before).

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/NRR_UMA'
    PKG = '/content/SAC_broadened_package'
    if not os.path.isdir(PKG):
        subprocess.run(['unzip', '-q', '-o', f'{WORK}/SAC_broadened_package.zip', '-d', '/content/'], check=True)
else:
    WORK = os.path.abspath('NRR_UMA_run')
    PKG = os.path.abspath(os.environ.get('SAC_PKG', 'SAC_broadened_package'))
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, PKG)
import json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ase.io import read
from ase.visualize.plot import plot_atoms
import nrr_mlip as nm
man = pd.read_csv(f'{PKG}/manifest.csv')
print(man[man.kind == 'bare'].groupby('family').size())
fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
for x, s in zip(ax, ['Fe@N2C2-PEN', 'Fe@C4', 'Fe@SV-N3', 'Fe@SV-C3']):
    plot_atoms(read(f'{PKG}/structures/{s}/N2_end.vasp'), x, radii=0.4, rotation='-75x'); x.set_title(s); x.axis('off')
plt.show()

## 3. Load the model
`TASK = 'omat'` was the best of the four settings in the validation run (no D3).

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TASK = 'omat'
if MODEL == 'uma':
    from fairchem.core import pretrained_mlip, FAIRChemCalculator
    predictor = pretrained_mlip.get_predict_unit('uma-s-1p1', device=DEVICE)
    # Later, for a FINE-TUNED model: load your checkpoint here instead (see the fairchem fine-tuning docs),
    # then keep everything else the same.
    calc = FAIRChemCalculator(predictor, task_name=TASK)
else:
    from mace.calculators import mace_mp
    TASK = 'mace'
    calc = mace_mp(model='medium-mpa-0', default_dtype='float32', device=DEVICE)
OUT = f'{WORK}/broadened_{TASK}'
os.makedirs(OUT, exist_ok=True)
print('results ->', OUT)

## 4. References (≈ 5 min): gas molecules, the 10 new supports, the 28 bulk metals

In [ ]:
TEST_MODE = False      # True = 2 catalysts only (to check the notebook)
t0 = time.time()
for _, r in man[man.tier == '0_reference'].iterrows():
    if TEST_MODE and r.kind == 'bulk' and r.system not in ('Fe', 'Mo'):
        continue
    nm.run_simple(r.kind, r.system, f'{PKG}/{r.file}', calc, OUT,
                  cell=(r.kind == 'bulk'), fmax=0.02 if r.kind == 'bulk' else 0.05)
print(f'references done in {time.time()-t0:.0f} s')

## 5. Relax the 280 catalysts (in tiers, so a stopped session still leaves complete tiers)
The cell first relaxes every catalyst for tier **core**, then adds tier **distal**, then tier **full**. It stops at the tier set by `RUN`.

| RUN | adds | gives | time on a T4 |
|---|---|---|---|
| `'core'` | bare, *H, *N2 end-on, *NNH end-on, *NH2 | Ef, ΔG*H, first step | about 2.2 h |
| `'distal'` | + *NNH2, *N, *NH, *NH3 | full distal U_L | about 2 h more (4.2 h total) |
| `'full'` | + side-on *N2/*NNH, alternating *NHNH, *NHNH2, *NH2NH2 | alternating pathway, side-on | about 3 h more (7 h total) |

**Over the daily GPU limit?** Nothing is lost. Every finished structure is already on Drive. Come back the next day, run all cells again, and it continues where it stopped. `RUN = 'distal'` (the default) fits in one day and gives everything needed for stability, U_L and HER.

In [ ]:
RUN = 'distal'         # 'core' | 'distal' | 'full'
CORE = ['H', 'N2_end', 'NNH_end', 'NH2']
DISTAL = CORE + ['NNH2', 'N', 'NH', 'NH3']
FULL = DISTAL + ['N2_side', 'NNH_side', 'NHNH', 'NHNH2', 'NH2NH2']
TIERS = [('core', CORE), ('distal', DISTAL), ('full', FULL)]
TIERS = TIERS[:[t for t, _ in TIERS].index(RUN) + 1]
SACS = sorted(man[man.kind == 'bare'].system)
if TEST_MODE:
    SACS = ['Fe@N2C2-PEN', 'Fe@SV-N3']
for label, ads_set in TIERS:
    t0 = time.time()
    for i, sac in enumerate(SACS):
        nm.run_sac(sac, f'{PKG}/structures', ads_set, calc, OUT, rebuild_ads=True)
        if (i + 1) % 10 == 0 or i + 1 == len(SACS):
            el = (time.time() - t0) / 60
            print(f'tier {label}: {i+1}/{len(SACS)} catalysts, {el:.0f} min, ~{el/(i+1)*(len(SACS)-i-1):.0f} min left in this tier', flush=True)
    print(f'tier {label} complete', flush=True)

## 6. Results table (zero-shot UMA, rough)
`status_*` columns flag intermediates that left the metal or broke, and `status_bare` flags a metal that left its site.
**Ef and ΔG\*H are rough trends only. Do not report ΔE/ΔG of the N-intermediates as DFT-quality numbers.**

In [ ]:
ads_df, sac_df = nm.analyse(f'{OUT}/results.jsonl', refs='computed')
sac_df['family'] = sac_df.Support.map(man.drop_duplicates('Support').set_index('Support').family)
sac_df['model'] = f'{MODEL}:{TASK} zero-shot'
sac_df.to_csv(f'{WORK}/broadened_SAC_results_{TASK}.csv', index=False)
ads_df.to_csv(f'{WORK}/broadened_structure_energies_{TASK}.csv', index=False)
print('saved to Drive:', f'broadened_SAC_results_{TASK}.csv')
cols = [c for c in ['SAC', 'family', 'status_bare', 'Ef_eV', 'dG_H*', 'dE_N2_end', 'dE_NNH_end', 'dE_NH2'] if c in sac_df]
print(sac_df[cols].round(2).to_string(index=False)[:6000])
print('\nmetal left its site:', (sac_df.status_bare != 'ok').sum(), 'of', len(sac_df))

In [ ]:
# Ef by support: which environments hold the metal best (rough trend)
ok = sac_df[sac_df.status_bare == 'ok']
piv = ok.pivot_table(index='Metal', columns='Support', values='Ef_eV')
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(piv.values, cmap='RdBu', vmin=-4, vmax=4, aspect='auto')
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=45, ha='right')
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
plt.colorbar(im, label='Ef (eV) - blue = metal prefers the site over its bulk')
ax.set_title(f'{MODEL}:{TASK} zero-shot formation energy (rough trend)'); fig.tight_layout()
fig.savefig(f'{WORK}/broadened_Ef_map_{TASK}.png', dpi=150); plt.show()

## 7. Export the UMA-relaxed structures (starting geometries for DFT / fine-tuning)
Writes `My Drive/NRR_UMA/UMA_relaxed_POSCARs.zip`. Give the folders listed in `dft_request_candidates.csv` to whoever
runs DFT (Pathak sir's group). Starting from these geometries, DFT relaxations need far fewer steps.

In [ ]:
req = pd.read_csv(f'{PKG}/dft_request_candidates.csv')
print(req.purpose.value_counts())
dest = '/content/UMA_relaxed_POSCARs' if IN_COLAB else os.path.join(WORK, 'UMA_relaxed_POSCARs')
n = nm.export_relaxed_poscars(OUT, dest)
import shutil
shutil.make_archive(os.path.join(WORK, 'UMA_relaxed_POSCARs'), 'zip', dest)
print(n, 'relaxed structures exported ->', os.path.join(WORK, 'UMA_relaxed_POSCARs.zip'))

**Send back:** `broadened_SAC_results_*.csv`, `broadened_structure_energies_*.csv`, `broadened_Ef_map_*.png`.
Also send the two `results.jsonl` files from the **first** run (`results_oc20/` and `results_omat/`), so the validation can be checked in detail.